In [2]:
import sys
import os
sys.path.append(os.path.abspath(".."))

from UniversalFunctions import *

In [ ]:
# Functions

# Get the pokemon number
def CheckPokedexNumber(number, minNumber, maxNumber):
    return number is not None and minNumber <= number <= maxNumber

# Get the pokemon's pokedex type
def GetLegalDexPokemon(pokemon, rules):
    pokedexRules = rules.get("allowed_regional_dexes", {})
    
    paldeaRules = pokedexRules.get("paldea")
    if paldeaRules and CheckPokedexNumber(
        pokemon.get("paldea_dex"),
        paldeaRules["min"],
        paldeaRules["max"]
    ): return "paldea_dex"
    
    kitakamiRules = pokedexRules.get("kitakami")
    if kitakamiRules and CheckPokedexNumber(
        pokemon.get("kitakami_dex"),
        kitakamiRules["min"],
        kitakamiRules["max"]
    ): return "kitakami_dex"
    
    blueberryRules = pokedexRules.get("blueberry")
    if blueberryRules and CheckPokedexNumber(
        pokemon.get("blueberry_dex"),
        blueberryRules["min"],
        blueberryRules["max"]
    ): return "blueberry_dex"
    
    if pokemon["name"] in set(rules.get("extra_allowed_pokemon", [])): return "extra_allowed"
    
    return None

# Exclude extra forms from pokemon so the AI does not use those
def ExcludePokemon(name, pokemon):
    if pokemon.get("is_mega", False):
        return True
    if pokemon.get("is_battle_only", False):
        return True
    
    excludedForms = [
        "-gmax",
        "-starter",
        "-mega",
        "-mega-x",
        "-mega-y",
        "-totem",
        "-totem-disguised",
        "-totem-busted"
    ]
    
    for form in excludedForms:
        if name.endswith(form):
            return True
        
    excludedExactForm = [
        "mimikyu-busted",
        "wishiwashi-school",
        "eiscue-noice",
        "morpeko-hangry",
        "palafin-hero"
    ]
    
    return name in excludedExactForm

# Function to format names so all pokemon variants are the same
def FormatPokemonName(name):
    nameMap = {
        "mimikyu-disguised": "mimikyu",
        "mimikyu-busted": "mimikyu",
        "mimikyu-totem-disguised": "mimikyu",
        "mimikyu-totem-busted": "mimikyu",
        "eiscue-ice": "eiscue",
        "eiscue-noice": "eiscue",
        "morpeko-full-belly": "morpeko",
        "morpeko-hangry": "morpeko",
        "wishiwashi-solo": "wishiwashi",
        "wishiwashi-school": "wishiwashi",
        "palafin-zero": "palafin",
        "palafin-hero": "palafin",
        "basculin-red-striped": "basculin",
        "tornadus-incarnate": "tornadus",
        "thundurus-incarnate": "thundurus",
        "landorus-incarnate": "landorus",
        "indeedee-male": "indeedee-m",
        "indeedee-female": "indeedee-f",
        "basculegion-male": "basculegion-m",
        "basculegion-female": "basculegion-f",
        "meowstic-male": "meowstic-m",
        "meowstic-female": "meowstic-f",
        "oinkologne-male": "oinkologne-m",
        "oinkologne-female": "oinkologne-f"
    }

    return nameMap.get(name, name)

# Makes sure to pick the original and not a variant
def ChooseBetterPokemon(existingPokemon, newPokemon):
    if newPokemon.get("is_default_form", False) and not existingPokemon.get("is_default_form", False):
        return newPokemon

    if existingPokemon.get("is_default_form", False) and not newPokemon.get("is_default_form", False):
        return existingPokemon

    if newPokemon.get("form_order", 9999) < existingPokemon.get("form_order", 9999):
        return newPokemon

    if existingPokemon.get("form_order", 9999) < newPokemon.get("form_order", 9999):
        return existingPokemon

    if newPokemon.get("order", 999999) < existingPokemon.get("order", 999999):
        return newPokemon

    return existingPokemon

# Create the regulation pool of valid pokemon
def CreateRegulationPokemon(pokemonData, rules):
    restricted = set(rules.get("restricted_pokemon", []))
    regulationPool = {}
    
    for pokemon in pokemonData:
        legalDexPokemon = GetLegalDexPokemon(pokemon, rules)
        if legalDexPokemon is None:
            continue
        
        name = pokemon["name"]
        if ExcludePokemon(name, pokemon):
            continue
        
        formattedName = FormatPokemonName(name)
        isRestricted = formattedName in restricted or name in restricted
        
        candidatePokemon = {
            "name": formattedName,
            "raw_name": name,
            "species_name": pokemon.get("species_name", ""),
            "generation": pokemon.get("generation", ""),
            "type": pokemon.get("type", []),
            "stats": pokemon.get("stats", {}),
            "bst": pokemon.get("bst", 0),
            "abilities": pokemon.get("abilities", []),
            "moveset": pokemon.get("moveset", []),
            "move_details": pokemon.get("move_details", {}),
            "gender_rate": pokemon.get("gender_rate", -1),
            "paldea_dex": pokemon.get("paldea_dex"),
            "kitakami_dex": pokemon.get("kitakami_dex"),
            "blueberry_dex": pokemon.get("blueberry_dex"),
            "is_default_form": pokemon.get("is_default_form", False),
            "is_battle_only": pokemon.get("is_battle_only", False),
            "is_mega": pokemon.get("is_mega", False),
            "form_order": pokemon.get("form_order", 0),
            "order": pokemon.get("order", 0),
            "legal_regulation_i": True,
            "restricted_regulation_i": isRestricted,
            "legal_source": legalDexPokemon,
            "format_name": rules.get("name", "Unknown Format"),
            "regulation_code": rules.get("regulation_code", "")
        }
        
        if formattedName not in regulationPool:
            regulationPool[formattedName] = candidatePokemon
        else:
            regulationPool[formattedName] = ChooseBetterPokemon(regulationPool[formattedName], candidatePokemon)
    
    regulationPool = sorted(regulationPool.values(), key=lambda p: p["name"])
    return regulationPool

In [ ]:
# Create the json files of the regulation pokemon

pokemonDataPath = "../../data/pokemon_data.json"
regulationPath = "../../data/regulation_i_rules.json"
outputPath = "../../data/pokemon_regulation_i.json"

pokemonData = LoadJSON(pokemonDataPath)
regulationRules = LoadJSON(regulationPath)

regulationPool = CreateRegulationPokemon(pokemonData, regulationRules)

WriteJSON(outputPath, regulationPool)

print(f"Saved {len(regulationPool)} legal Pokemon to {outputPath}")

Saved 835 legal Pokemon to ../../data/pokemon_regulation_i.json
